In [ ]:
# 1. Cài đặt thư viện (Bản mới nhất, không ép version numpy để tránh lỗi)
!pip install -U ultralytics

In [3]:
import json
import os
import cv2
import numpy as np
import random
import shutil
from tqdm import tqdm
from ultralytics import YOLO
from pathlib import Path

# ================= CẤU HÌNH =================
DATASET_ROOT = "/kaggle/input/zalo-video/train"  
OUTPUT_DIR = "dataset_finetune"   

# Tham số lấy mẫu
SAMPLING_RATE = 20                
CONF_THRESHOLD = 0.2              
IOU_THRESH_NEG = 0.05             

# --- CÂN BẰNG DỮ LIỆU ---
MAX_NEG_PER_POS = 1               
NUM_BG_PER_FRAME = 1              

# Đường dẫn cụ thể
ANN_PATH = os.path.join(DATASET_ROOT, "annotations", "annotations.json")
SAMPLES_DIR = os.path.join(DATASET_ROOT, "samples")

# ================= HÀM HỖ TRỢ =================
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area > 0 else 0

def save_crop(frame, box, save_dir, prefix):
    x1, y1, x2, y2 = map(int, box)
    h, w, _ = frame.shape
    # Clip tọa độ
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    
    if x2 > x1 and y2 > y1:
        crop = frame[y1:y2, x1:x2]
        if crop.shape[0] < 10 or crop.shape[1] < 10: return
        filename = f"{prefix}_{x1}_{y1}.jpg"
        cv2.imwrite(os.path.join(save_dir, filename), crop)

def random_crop_background(frame, gt_boxes, num_crops=1):
    h, w, _ = frame.shape
    crops = []
    attempts = 0
    while len(crops) < num_crops and attempts < 50:
        attempts += 1
        cw = random.randint(50, 200)
        ch = random.randint(50, 200)
        if cw >= w or ch >= h: continue
        cx = random.randint(0, w - cw)
        cy = random.randint(0, h - ch)
        rand_box = [cx, cy, cx+cw, cy+ch]
        
        is_clean = True
        for gt in gt_boxes:
            if compute_iou(rand_box, gt) > 0.01:
                is_clean = False; break
        if is_clean: crops.append(rand_box)
    return crops

# ================= MAIN PROCESS =================
def main():
    # Setup thư mục (Xóa cũ tạo mới để tránh lẫn lộn)
    if os.path.exists(OUTPUT_DIR):
        shutil.rmtree(OUTPUT_DIR)
    
    for sub in ['anchors', 'positives', 'negatives/hard', 'negatives/background']:
        Path(os.path.join(OUTPUT_DIR, sub)).mkdir(parents=True, exist_ok=True)

    # Load GT
    print("⏳ Đang đọc Annotation...")
    with open(ANN_PATH, 'r') as f:
        ann_data = json.load(f)
        if isinstance(ann_data, dict): ann_data = [ann_data]

    gt_map = {}
    for rec in ann_data:
        vid = rec['video_id']
        gt_map[vid] = {}
        for anno in rec.get('annotations', []):
            for b in anno.get('bboxes', []):
                gt_map[vid].setdefault(b['frame'], []).append([b['x1'], b['y1'], b['x2'], b['y2']])

    # Load YOLO
    print("⏳ Đang load YOLO...")
    model = YOLO("yolov8n.pt") 

    # Duyệt video
    video_folders = sorted(os.listdir(SAMPLES_DIR))
    
    total_pos = 0
    total_neg = 0
    
    for vid_id in tqdm(video_folders, desc="Processing"):
        vid_path = os.path.join(SAMPLES_DIR, vid_id)
        video_file = os.path.join(vid_path, "drone_video.mp4")
        if not os.path.exists(video_file): continue

        # Copy Anchors
        ref_src_dir = os.path.join(vid_path, "object_images")
        if os.path.exists(ref_src_dir):
            for i, img_name in enumerate(os.listdir(ref_src_dir)):
                src = os.path.join(ref_src_dir, img_name)
                dst = os.path.join(OUTPUT_DIR, 'anchors', f"{vid_id}_ref_{i}.jpg")
                shutil.copy(src, dst)

        pos_save_dir = os.path.join(OUTPUT_DIR, 'positives', vid_id)
        os.makedirs(pos_save_dir, exist_ok=True)

        cap = cv2.VideoCapture(video_file)
        frame_idx = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            if frame_idx % SAMPLING_RATE == 0:
                current_gt = gt_map.get(vid_id, {}).get(frame_idx, [])
                num_pos_in_frame = len(current_gt)
                
                # --- 1. LẤY POSITIVE ---
                for i, box in enumerate(current_gt):
                    save_crop(frame, box, pos_save_dir, prefix=f"f{frame_idx}_p{i}")
                    total_pos += 1

                # --- 2. LẤY NEGATIVE ---
                # HACK: Dùng try-except để bỏ qua lỗi nếu YOLO gặp vấn đề với frame cụ thể
                try:
                    # Lấy Hard Negative có giới hạn
                    limit_hard_neg = max(1, num_pos_in_frame) * MAX_NEG_PER_POS
                    count_hard_neg = 0
                    
                    # Truyền trực tiếp frame (Numpy array) cho YOLO
                    # YOLOv8 tự động xử lý BGR/RGB
                    results = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)
                    
                    for res in results:
                        for box in res.boxes.xyxy.cpu().numpy():
                            if count_hard_neg >= limit_hard_neg: break 
                                
                            max_iou = 0
                            for gt in current_gt:
                                max_iou = max(max_iou, compute_iou(box, gt))
                            
                            if max_iou < IOU_THRESH_NEG:
                                save_dir = os.path.join(OUTPUT_DIR, 'negatives/hard')
                                save_crop(frame, box, save_dir, prefix=f"{vid_id}_f{frame_idx}_h{count_hard_neg}")
                                count_hard_neg += 1
                                total_neg += 1
                except Exception as e:
                    # Nếu frame lỗi thì bỏ qua, không dừng chương trình
                    pass

                # --- 3. LẤY BACKGROUND NEGATIVE ---
                bg_boxes = random_crop_background(frame, current_gt, num_crops=NUM_BG_PER_FRAME)
                for i, box in enumerate(bg_boxes):
                    save_dir = os.path.join(OUTPUT_DIR, 'negatives/background')
                    save_crop(frame, box, save_dir, prefix=f"{vid_id}_f{frame_idx}_bg{i}")
                    total_neg += 1

            frame_idx += 1
        cap.release()

    print(f"\n✅ Hoàn tất! Pos: {total_pos}, Neg: {total_neg}")
    if total_pos > 0:
        print(f"📊 Tỉ lệ thực tế: 1 : {total_neg/total_pos:.2f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
⏳ Đang đọc Annotation...
⏳ Đang load YOLO...


Processing: 100%|██████████| 14/14 [02:42<00:00, 11.58s/it]


✅ Hoàn tất! Pos: 1005, Neg: 4726
📊 Tỉ lệ thực tế: 1 : 4.70


In [ ]:
if __name__ == "__main__":
    main()